In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import average_precision_score
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.model_selection import StratifiedKFold
import sys
import scrapbook as sb

sys.path.append('..')
from utils import reshape_to_numpy, interpolate_missing_values, denoise_data
from ts2vec.ts2vec import TS2Vec

In [ ]:
seed = 1
accel_cutoff = 0.4
accel_order = 30
gyro_cutoff = 0.6
gyro_order = 20
input_dims = 6
output_dims = 32
hidden_dims = 64
depth = 6
batch_size = 32
max_train_length = 600
n_epochs = 300
patience = 15
patience_delta = 1e-3
lr = 0.001
n_splits = 5

In [ ]:
rng = np.random.RandomState(seed)

In [ ]:
data = pd.read_parquet("../data/GBG500.parquet")
data

In [ ]:
labels = pd.read_csv("../data/GBG500_labels.csv")
labels.columns = labels.columns.str.lower()
ride_order_df = pd.DataFrame({"ride_id": data["ride_id"].unique()})
labels_sorted = ride_order_df.merge(
    labels,
    on="ride_id",
    how="left"
)
labels_sorted

In [ ]:
data_np = reshape_to_numpy(
    data,
    features = ["ax", "ay", "az", "rx", "ry", "rz"],
    max_timestamps = 4800
)

data_np_clean = interpolate_missing_values(
    data_np,
    method='linear',
    limit=None
)

data_np_clean = denoise_data(
    data=data_np_clean,
    accel_indices=[0, 1, 2],
    gyro_indices=[3, 4, 5],
    accel_cutoff=accel_cutoff,
    accel_order=accel_order,
    gyro_cutoff=gyro_cutoff,
    gyro_order=gyro_order,
)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

class EarlyStoppingException(Exception):
    pass

y_true = (labels_sorted['label'] == 'Reckless').astype(int).values
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=rng.randint(1000))
scores = {k: np.full(len(y_true), np.nan) for k in ['lof', 'iso_forest', 'ocsvm']}

for fold, (train_idx, test_idx) in enumerate(skf.split(data_np_clean, y_true)):
    state = {'best_loss': float('inf'), 'patience_counter': 0}

    def after_epoch_callback(model, loss):
        if loss < state['best_loss'] - patience_delta:
            state['best_loss'] = loss
            state['patience_counter'] = 0
        else:
            state['patience_counter'] += 1
            if state['patience_counter'] >= patience:
                raise EarlyStoppingException()

    ts2vec = TS2Vec(
        input_dims=input_dims,
        output_dims=output_dims,
        hidden_dims=hidden_dims,
        depth=depth,
        device=device,
        lr=lr,
        batch_size=batch_size,
        max_train_length=max_train_length,
        after_epoch_callback=after_epoch_callback,
    )

    try:
        ts2vec.fit(data_np_clean[train_idx], n_epochs=n_epochs, verbose=False)
    except EarlyStoppingException:
        pass

    Z_train = ts2vec.encode(data_np_clean[train_idx], encoding_window='full_series')
    Z_test = ts2vec.encode(data_np_clean[test_idx], encoding_window='full_series')

    enc_scaler = StandardScaler()
    Z_train = enc_scaler.fit_transform(Z_train)
    Z_test = enc_scaler.transform(Z_test)

    lof = LocalOutlierFactor(n_neighbors=20, novelty=True)
    lof.fit(Z_train)
    scores['lof'][test_idx] = -lof.score_samples(Z_test)

    iso = IsolationForest(random_state=rng.randint(1000))
    iso.fit(Z_train)
    scores['iso_forest'][test_idx] = -iso.score_samples(Z_test)

    ocsvm = OneClassSVM(kernel='rbf')
    ocsvm.fit(Z_train)
    scores['ocsvm'][test_idx] = -ocsvm.decision_function(Z_test)

    print(f"Fold {fold+1}/{n_splits} done")

In [ ]:
for key, glue_name in [('lof', 'GBG500_ap_ts2vec_lof'),
                        ('iso_forest', 'GBG500_ap_ts2vec_iso_forest'),
                        ('ocsvm', 'GBG500_ap_ts2vec_ocsvm')]:
    ap = average_precision_score(y_true, scores[key])
    print(f"TS2Vec+{key} AP = {ap:.4f}")
    sb.glue(glue_name, float(ap))
